# Computational Trade-offs and Batching Benchmark

This notebook validates the claims made in the `Computational Trade-offs and Batching` section of the manuscript.
It provides a comprehensive measurement of inference speed and memory usage for **Prompt U-Net (v330/v332 architecture)**, **UniverSeg**, and **nnInteractive**.

**Settings evaluated:**
- CPU Execution (Single Batch) to demonstrate memory efficiency in resource-constrained environments.
- GPU Execution (Max Batch) to demonstrate throughput improvements via parallel compute saturation.

**Structures evaluated (40 2D slices each):**
- **Small (128x128):** Fits within the native tile size, zero tiling overhead for Prompt U-Net.
- **Medium (256x256):** Requires adaptive tiling for Prompt U-Net.
- **Large (512x512):** Requires extensive adaptive tiling for Prompt U-Net.


In [ ]:
# Run this cell ONLY if you are in Google Colab to setup the environment
import sys
import os

if 'google.colab' in sys.modules:
    print("Setting up Colab environment...")
    !git clone https://github.com/Machauer-P/prompt-unet.git
    %cd prompt-unet
    
    # Make sure we pull the large keras files
    !git lfs pull
    
    print("Installing requirements...")
    !pip install -r requirements_eval.txt > /dev/null
    !pip install tf-keras > /dev/null
    
    # --- FIX: Recreate the missing benchmark_models directory and clone the models ---
    !mkdir -p evaluation/benchmark_models
    
    print("Cloning and installing UniverSeg...")
    !git clone https://github.com/JJGO/UniverSeg.git evaluation/benchmark_models/UniverSeg
    !pip install -e evaluation/benchmark_models/UniverSeg > /dev/null
    
    print("Cloning and installing nnInteractive...")
    !git clone https://github.com/MIC-DKFZ/nnInteractive.git evaluation/benchmark_models/nnInteractive
    !pip install -e evaluation/benchmark_models/nnInteractive > /dev/null
    
    # Make sure nnInteractive is discoverable by python
    if os.path.abspath("evaluation/benchmark_models/nnInteractive") not in sys.path:
        sys.path.insert(0, os.path.abspath("evaluation/benchmark_models/nnInteractive"))
    
    PROJECT_ROOT = os.path.abspath(".")
else:
    PROJECT_ROOT = os.path.abspath("../..")
    
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    
print("Environment Ready!")

In [ ]:
import gc
import time
import psutil
import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from inference.predictor import PromptUNetPredictor
from evaluation.benchmark_models.UniverSeg.universeg import universeg
from huggingface_hub import snapshot_download
from nnInteractive.inference.inference_session import nnInteractiveInferenceSession

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

GPU_BATCH_SIZE_PUNET = 32
GPU_BATCH_SIZE_UNIVERSEG = 32
NUM_SLICES = 40

SIZES = {
    "Small (128x128)": (128, 128),
    "Medium (256x256)": (256, 256),
    "Large (512x512)": (512, 512)
}

ENVIRONMENTS = ["CPU (Batch=1)", "GPU (Max Batch)"]

# Set TF memory growth to prevent TF from allocating the entire 16GB immediately
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
# Initialize Models

print("Loading Prompt U-Net...")
# Using p_unet_330.keras as the representative final architecture model
punet_path = os.path.join(PROJECT_ROOT, "training", "p_unet_330.keras")
punet_predictor = PromptUNetPredictor(punet_path)

print("Loading UniverSeg...")
universeg_model = universeg(pretrained=False) # For timing, random weights are fine
universeg_model.eval()

print("Loading nnInteractive...")
model_path = snapshot_download(
    repo_id="nnInteractive/nnInteractive",
    allow_patterns=["nnInteractive_v1.0/*"],
    local_dir="./nninteractive_model"
)
nn_session = nnInteractiveInferenceSession(device=torch.device('cpu'), use_torch_compile=False, verbose=False)
nn_session.initialize_from_trained_model_folder(os.path.join(model_path, "nnInteractive_v1.0"))
nninteractive_model = nn_session.network
nninteractive_model.eval()
print("All models loaded!")


In [ ]:
def get_gpu_memory_tf():
    try:
        mem_info = tf.config.experimental.get_memory_info('GPU:0')
        return mem_info['peak'] / (1024**2)
    except:
        return 0

def get_gpu_memory_torch(device):
    if device.type == 'cuda':
        return torch.cuda.max_memory_allocated(device) / (1024**2)
    return 0

def clean_memory(env):
    gc.collect()
    tf.keras.backend.clear_session()
    if 'GPU' in env:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        try:
            tf.config.experimental.reset_memory_stats('GPU:0')
        except:
            pass


In [ ]:
def run_punet(env, H, W, num_slices=NUM_SLICES):
    device_name = '/CPU:0' if 'CPU' in env else '/GPU:0'
    batch_size = 1 if 'CPU' in env else GPU_BATCH_SIZE_PUNET
    
    x = np.random.randn(num_slices, H, W).astype(np.float32)
    p = np.zeros((num_slices, H, W, 2), dtype=np.float32)
    
    # Create a bounding box covering 80% of the image to force realistic tiling
    y_start, y_end = int(H * 0.1), int(H * 0.9)
    x_start, x_end = int(W * 0.1), int(W * 0.9)
    p[:, y_start:y_end, x_start:x_end, 1] = 1.0 
    
    clean_memory(env)
    
    # Warmup
    with tf.device(device_name):
        _ = punet_predictor.predict(x[:1], p[:1], batch_size=batch_size)
    
    clean_memory(env)
    
    start_time = time.perf_counter()
    with tf.device(device_name):
        _ = punet_predictor.predict(x, p, batch_size=batch_size)
    end_time = time.perf_counter()
    
    mem_mb = get_gpu_memory_tf() if 'GPU' in env else 0
    return (end_time - start_time) * 1000, mem_mb

def run_universeg(env, H, W, num_slices=NUM_SLICES):
    device = torch.device('cpu' if 'CPU' in env else 'cuda:0')
    batch_size = 1 if 'CPU' in env else GPU_BATCH_SIZE_UNIVERSEG
    
    model = universeg_model.to(device)
    
    x = torch.randn(num_slices, 1, H, W)
    sup_x = torch.randn(num_slices, 16, 1, 128, 128)
    sup_y = torch.zeros(num_slices, 16, 1, 128, 128)
    
    clean_memory(env)
    
    # Warmup
    with torch.no_grad():
        x_128 = torch.nn.functional.interpolate(x[:1].to(device), size=(128, 128), mode='bilinear', align_corners=False)
        _ = model(x_128, sup_x[:1].to(device), sup_y[:1].to(device))
        if 'GPU' in env: torch.cuda.synchronize()
        
    clean_memory(env)
    
    start_time = time.perf_counter()
    with torch.no_grad():
        for i in range(0, num_slices, batch_size):
            batch_x = x[i:i+batch_size].to(device)
            batch_sx = sup_x[i:i+batch_size].to(device)
            batch_sy = sup_y[i:i+batch_size].to(device)
            
            batch_x_128 = torch.nn.functional.interpolate(batch_x, size=(128, 128), mode='bilinear', align_corners=False)
            logits = model(batch_x_128, batch_sx, batch_sy)
            _ = torch.nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
            
    if 'GPU' in env: torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    mem_mb = get_gpu_memory_torch(device) if 'GPU' in env else 0
    model.cpu()
    return (end_time - start_time) * 1000, mem_mb

def run_nninteractive(env, H, W, num_slices=NUM_SLICES):
    device = torch.device('cpu' if 'CPU' in env else 'cuda:0')
    model = nninteractive_model.to(device)
    
    # nnInteractive uses 1 image + 7 interaction channels
    C_IN = 8
    dummy_input = torch.randn(1, C_IN, num_slices, H, W).to(device)
    
    clean_memory(env)
    
    # Warmup
    try:
        with torch.no_grad():
            _ = model(dummy_input)
            if 'GPU' in env: torch.cuda.synchronize()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            model.cpu()
            return np.nan, np.nan
        raise e
        
    clean_memory(env)
    
    start_time = time.perf_counter()
    try:
        with torch.no_grad():
            _ = model(dummy_input)
            if 'GPU' in env: torch.cuda.synchronize()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            model.cpu()
            return np.nan, np.nan
        raise e
        
    end_time = time.perf_counter()
    
    mem_mb = get_gpu_memory_torch(device) if 'GPU' in env else 0
    model.cpu()
    return (end_time - start_time) * 1000, mem_mb


In [ ]:
# Run Benchmarks
results = []

for env in ENVIRONMENTS:
    print(f"\n--- Evaluating on {env} ---")
    for size_name, (H, W) in SIZES.items():
        print(f"  Structure Size: {size_name}")
        
        # Prompt U-Net
        t_punet, mem_punet = run_punet(env, H, W)
        results.append({"Environment": env, "Size": size_name, "Model": "Prompt U-Net", "Time (ms)": t_punet, "Memory (MB)": mem_punet})
        print(f"    Prompt U-Net  : {t_punet:.1f} ms, {mem_punet:.1f} MB")
        
        # UniverSeg
        t_uni, mem_uni = run_universeg(env, H, W)
        results.append({"Environment": env, "Size": size_name, "Model": "UniverSeg", "Time (ms)": t_uni, "Memory (MB)": mem_uni})
        print(f"    UniverSeg     : {t_uni:.1f} ms, {mem_uni:.1f} MB")
        
        # nnInteractive
        t_nn, mem_nn = run_nninteractive(env, H, W)
        results.append({"Environment": env, "Size": size_name, "Model": "nnInteractive", "Time (ms)": t_nn, "Memory (MB)": mem_nn})
        print(f"    nnInteractive : {t_nn:.1f} ms, {mem_nn:.1f} MB" if not np.isnan(t_nn) else "    nnInteractive : OOM")

df = pd.DataFrame(results)
display(df)


In [ ]:
# Visualization

# Filter out OOM (NaN) for plotting
df_plot = df.dropna()

# Inference Time Plot
g = sns.catplot(
    data=df, kind="bar",
    x="Size", y="Time (ms)", hue="Model", col="Environment",
    palette=["#4c72b0", "#dd8452", "#55a868"],
    height=6, aspect=1.2, sharey=False
)
g.fig.suptitle("Volume Inference Time (40 slices) - Lower is Better", y=1.05, fontweight="bold")
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_ylabel("Time (ms) [Log Scale]")
    
    # Add annotations
    for p in ax.patches:
        height = p.get_height()
        if not np.isnan(height) and height > 0:
            ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, rotation=90, xytext=(0, 5),
                        textcoords='offset points')

plt.show()

# Memory Plot (Only valid for GPU)
df_gpu = df[df["Environment"] == "GPU (Max Batch)"].copy()
plt.figure(figsize=(8, 6))
sns.barplot(data=df_gpu, x="Size", y="Memory (MB)", hue="Model", palette=["#4c72b0", "#dd8452", "#55a868"])
plt.title("Peak GPU Memory Usage", fontweight="bold")
plt.ylabel("Memory (MB)")

# Add annotations
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10)

plt.show()
